In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('C:/Users/karti/Desktop/Proj-1/customer_shopping_behavior.csv')
df

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3895,3896,40,Female,Hoodie,Clothing,28,Virginia,L,Turquoise,Summer,4.2,No,2-Day Shipping,No,No,32,Venmo,Weekly
3896,3897,52,Female,Backpack,Accessories,49,Iowa,L,White,Spring,4.5,No,Store Pickup,No,No,41,Bank Transfer,Bi-Weekly
3897,3898,46,Female,Belt,Accessories,33,New Jersey,L,Green,Spring,2.9,No,Standard,No,No,24,Venmo,Quarterly
3898,3899,44,Female,Shoes,Footwear,77,Minnesota,S,Brown,Summer,3.8,No,Express,No,No,24,Venmo,Weekly


In [3]:
# df.isnull().sum()
# df.groupby('Category')['Review Rating'].median()

In [4]:
# we had nulls in review rating so filled them with each category rating median to make robust distribution in data
# transform calculates per group but keeps the same number of rows 
df['Review Rating'] = df.groupby('Category')[['Review Rating']].transform(lambda x : x.fillna(x.median()))
# df.isnull().sum()



In [5]:

# snake casing the columns
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ','_')

df = df.rename(columns={"purchase_amount_(usd)": "purchase_amount"}) 

# df.columns

In [6]:
# df['age_group'] = pd.cut(
#     df['Age'],
#     bins=[0, 18, 35, 50, 100],
#     labels=['0-18', '19-35', '36-50', '51+']
# )

In [7]:
labels = ['Young Adult', 'Adult','Middle-aged', 'Senior']

df['age_group'] = pd.qcut(df['age'],q=4,labels=labels)
# df[['age','age_group']]

In [8]:
# we have monthly, quartly in text but to analyse and visualise its good to make them nums
# frequency of purchase means how frequently he buys eg every 7 days 
# df['frequency_of_purchases'].unique()

frequency_mapping = {
    'Fortnightly' : 14, 
    'Weekly' : 7,
    'Annually' : 365,
    'Quarterly' : 90,
    'Bi-Weekly' : 14,
    'Monthly' : 30,
    'Every 3 Months' : 90
}

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

# df[['frequency_of_purchases','purchase_frequency_days']]

In [9]:
(df['discount_applied'] == df['promo_code_used']).all()
# both signifies the same meaning and have same values in every row, so removing one row

df = df.drop('promo_code_used',axis=1)

In [10]:
# df

In [11]:
# !pip install pymysql sqlalchemy

In [15]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# MySQL connection
# username = "root"
# password = "Kar@mysql1310"
# host = "localhost"
# port = "3306"
# database = "customer_behavior"

# engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")
# since i have special character in password as @, so it interprets so using different way

connection_url = URL.create(
    "mysql+pymysql",
    username="root",
    password="Kar@mysql1310",
    host="localhost",
    port=3306,
    database="customer_behavior"
)

engine = create_engine(connection_url)

# Write DataFrame to MySQL
table_name = "customer"   # choose any table name
df.to_sql(table_name, engine, if_exists="replace", index=False)

# Read back sample
pd.read_sql("SELECT * FROM customer LIMIT 5;", engine)

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,age_group,purchase_frequency_days
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,14,Venmo,Fortnightly,Middle-aged,14
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,2,Cash,Fortnightly,Young Adult,14
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,23,Credit Card,Weekly,Middle-aged,7
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,49,PayPal,Weekly,Young Adult,7
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,31,PayPal,Annually,Middle-aged,365
